# Semana del 25 de noviembre de 2026 · Vista 3×3, tres figuras de colores y sampler 1

En `distinct` hay **tres obstáculos**, todos con forma y color diferentes:

- Un **cuadrado verde** y un **triángulo morado** en las dos habitaciones laterales. Se sortea qué figura recibe cada habitación, como en el experimento anterior.
- Un **rombo amarillo** en la habitación del botón.

Al comenzar cada episodio se sortea la casilla de cada figura **dentro de su habitación**. Permanecen quietas durante toda la partida. Ninguna aparece en los pasillos, sobre el botón, en sus cuatro casillas adyacentes ni en las esquinas de las galletas. Se mantiene libre el pasillo vertical de acceso al botón. La verificación comprueba las 5.832 combinaciones de posiciones y que ninguna aísle casillas transitables.

Se mantiene la configuración del 3×3 con sampler: actor de Dreamer, `buffer.reward_slices=1`, 500.000 pasos, semilla 0, decoder, train ratio 128, imagen 64×64 y evaluación cada 25.000 pasos con cinco partidas. El sampler repite una secuencia con recompensa por batch cuando hay alguna guardada.

Elige `distinct` para esta prueba. Los controles opcionales son `same` (tres cuadrados verdes) y `none` (sin obstáculos).

Resultados: `MyDrive/galleta/semana25/tres_obstaculos_colores3x3/{modo}_sampler1_s{semilla}`. Esta variante comienza un entrenamiento nuevo; las siguientes sesiones retoman su propio checkpoint. La sección 6 compara sus métricas con el 3×3 anterior de dos obstáculos con sampler.


## 1. Preparar la GPU y montar Drive

Selecciona una GPU L4 en Colab. Drive conserva los checkpoints, el buffer y las métricas entre sesiones.

In [ ]:
import subprocess
from google.colab import drive

subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    check=True,
)
drive.mount("/content/drive")

## 2. Preparar el código y comprobar el entorno

Se descarga una revisión fija en una copia propia de Colab y se aplica el cambio incluido en este notebook. La verificación comprueba la aleatoriedad, la estabilidad durante el episodio, las semillas y el acceso a los objetivos. Luego se muestra una vista previa de tres episodios.


In [ ]:
#@title Descargar el código e instalar las dependencias
from pathlib import Path
import subprocess
import sys

REPO = Path("/content/r2dreamer-cookie-semana25-tres-colores-vista3-sampler1")
URL = "https://github.com/FabriRandon/r2dreamer-cookie.git"
REVISION = "ac26925457a7188b4668c526d5b2affad2ac6f4f"
CAMBIOS = '--- a/envs/cookie.py\n+++ b/envs/cookie.py\n@@ -7,6 +7,9 @@\n # MiniGrid\'s action indices for moving forward and for toggling the object in\n # front of the agent.\n FORWARD, TOGGLE = 2, 5\n+\n+# This variant lives only in the November 25 three-obstacle experiments.\n+OBSTACLE_LAYOUT = "three_colored_per_episode_v1"\n \n # Variants of the cookie domain (cookie_env.envs.ThreeRooms), mirroring the\n # ones used in the JAX DreamerV3 study this fork replicates. "full" shows the\n@@ -45,21 +48,22 @@\n     corners = [(x, y) for x in (cx - hallway - 2, cx + hallway + 2) for y in (cy - 2, cy + 2)]\n \n     class Obstacle(Wall):\n-        def __init__(self, triangle):\n-            super().__init__("green")\n-            self.triangle = triangle\n+        def __init__(self, shape, color):\n+            super().__init__(color)\n+            self.shape = shape\n \n         def encode(self):\n-            # Grid.render_tile caches by encoding, so each shape needs its own key.\n+            # MiniGrid caches rendered tiles by object type, color and state.\n             kind, color, _ = super().encode()\n-            return kind, color, int(self.triangle)\n+            return kind, color, ("square", "triangle", "diamond").index(self.shape)\n \n         def render(self, img):\n-            shape = (\n-                point_in_triangle((0.5, 0.08), (0.9, 0.88), (0.1, 0.88))\n-                if self.triangle\n-                else point_in_rect(0.22, 0.78, 0.22, 0.78)\n-            )\n+            if self.shape == "triangle":\n+                shape = point_in_triangle((0.5, 0.08), (0.9, 0.88), (0.1, 0.88))\n+            elif self.shape == "diamond":\n+                shape = lambda x, y: abs(x - 0.5) + abs(y - 0.5) <= 0.4\n+            else:\n+                shape = point_in_rect(0.22, 0.78, 0.22, 0.78)\n             fill_coords(img, shape, COLORS[self.color])\n \n     class Rooms(ThreeRooms):\n@@ -72,8 +76,41 @@\n             self._generate_hallway(cx, cy - hallway, cx, cy)\n             self.put_obj(Button("blue", self.spawn_cookie), cx, cy - hallway)\n             if obstacles != "none":\n-                self.put_obj(Obstacle(triangle=False), cx - hallway, cy - 1)\n-                self.put_obj(Obstacle(triangle=obstacles == "distinct"), cx + hallway, cy + 1)\n+                # Sample once per reset, using Gymnasium\'s seeded RNG. Both\n+                # rooms have the same candidate offsets: neither the position\n+                # nor the shape has a permanent left/right meaning. Keep the\n+                # central aisle and all four cookie corners free.\n+                positions = []\n+                for room_x in (cx - hallway, cx + hallway):\n+                    candidates = [\n+                        (room_x + dx, cy + dy)\n+                        for dx in range(-2, 3)\n+                        for dy in (-2, -1, 1, 2)\n+                        if (room_x + dx, cy + dy) not in corners\n+                    ]\n+                    positions.append(candidates[int(self.np_random.integers(len(candidates)))])\n+                # Draw in both modes so same/distinct use identical layouts\n+                # for the same seed and episode. Do not consume the global\n+                # Python RNG, which independently chooses cookie corners.\n+                triangle_room = int(self.np_random.integers(2))\n+                for room, (x, y) in enumerate(positions):\n+                    triangle = obstacles == "distinct" and room == triangle_room\n+                    shape = "triangle" if triangle else "square"\n+                    color = "purple" if triangle else "green"\n+                    self.put_obj(Obstacle(shape, color), x, y)\n+                # The button room has a vertical entrance. Keep that entire\n+                # aisle and all four tiles adjacent to the button free.\n+                button_y = cy - hallway\n+                candidates = [\n+                    (cx + dx, button_y + dy)\n+                    for dx in (-2, -1, 1, 2)\n+                    for dy in range(-2, 3)\n+                    if abs(dx) + abs(dy) > 1\n+                ]\n+                x, y = candidates[int(self.np_random.integers(len(candidates)))]\n+                # same is the visual control: three identical green squares.\n+                shape, color = ("diamond", "yellow") if obstacles == "distinct" else ("square", "green")\n+                self.put_obj(Obstacle(shape, color), x, y)\n             self.agent_pos = self.agent_start_pos\n             self.agent_dir = self.agent_start_dir\n \n'


def git(*args, **kwargs):
    return subprocess.run(["git", *args], cwd=REPO, **kwargs)

REPO.mkdir(parents=True, exist_ok=True)
if not (REPO / ".git").is_dir():
    git("init", "--quiet", check=True)
actual = git("rev-parse", "--verify", "HEAD", capture_output=True, text=True)
if actual.returncode != 0:
    git("fetch", "--quiet", "--depth=1", URL, REVISION, check=True)
    git("checkout", "--quiet", "--detach", "FETCH_HEAD", check=True)
elif actual.stdout.strip() != REVISION:
    raise RuntimeError(f"Revisión inesperada en {REPO}. Revisa esa copia antes de continuar.")

if git("apply", "--reverse", "--check", "-", input=CAMBIOS, text=True, capture_output=True).returncode:
    git("apply", "--check", "-", input=CAMBIOS, text=True, check=True)
    git("apply", "-", input=CAMBIOS, text=True, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
PYTHON = REPO / ".venv/bin/python"
if not PYTHON.exists():
    subprocess.run([*UV, "venv", "--python", "3.11", ".venv"], cwd=REPO, check=True)
subprocess.run([*UV, "pip", "install", "--python", str(PYTHON), "-e", ".[cookie]"], cwd=REPO, check=True)

In [ ]:
#@title Verificar tres figuras, colores, semillas y todos los caminos
comprobacion = '"""Validate the three colored obstacles without training or using a GPU."""\n\nfrom collections import deque\nimport importlib.util\nfrom itertools import product\nfrom pathlib import Path\nimport sys\n\n\ndef figuras(raw):\n    return tuple(\n        (x, y, obj.shape, obj.color)\n        for x in range(raw.width)\n        for y in range(raw.height)\n        if (obj := raw.grid.get(x, y)) is not None and hasattr(obj, "shape")\n    )\n\n\ndef alcanzables(raw):\n    seen, queue = {(8, 8)}, deque([(8, 8)])\n    while queue:\n        x, y = queue.popleft()\n        for pos in ((x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)):\n            if pos in seen or not (0 <= pos[0] < raw.width and 0 <= pos[1] < raw.height):\n                continue\n            obj = raw.grid.get(*pos)\n            if obj is None or obj.can_overlap():\n                seen.add(pos)\n                queue.append(pos)\n    walkable = {\n        (x, y) for x in range(raw.width) for y in range(raw.height)\n        if (obj := raw.grid.get(x, y)) is None or obj.can_overlap()\n    }\n    assert seen == walkable, "Una figura deja casillas transitables aisladas"\n    return seen\n\n\ndef verificar(module):\n    assert module.OBSTACLE_LAYOUT == "three_colored_per_episode_v1", "Se cargó otra versión del entorno"\n    corners = {(1, 6), (1, 10), (15, 6), (15, 10)}\n    button_access = {(8, 2), (8, 4), (7, 3), (9, 3)}\n    side_positions = [\n        {(cx + dx, 8 + dy) for dx in range(-2, 3) for dy in (-2, -1, 1, 2)} - corners\n        for cx in (3, 13)\n    ]\n    center_positions = {\n        (8 + dx, 3 + dy) for dx in (-2, -1, 1, 2) for dy in range(-2, 3)\n        if abs(dx) + abs(dy) > 1\n    }\n    assert all(len(positions) == 18 for positions in (*side_positions, center_positions))\n\n    def secuencia(mode, view, seed):\n        env = module.Cookie(f"partial_hall5_view{view}", obstacles=mode, seed=seed)\n        layouts, cookies = [], []\n        try:\n            for _ in range(24):\n                obs = env.reset()\n                raw = env._env.unwrapped\n                assert obs["image"].shape == (64, 64, 3)\n                assert raw.agent_view_size == view\n                layout = figuras(raw)\n                assert len(layout) == (0 if mode == "none" else 3)\n                if layout:\n                    left, center, right = layout\n                    assert left[:2] in side_positions[0] and right[:2] in side_positions[1]\n                    assert center[:2] in center_positions\n                    if mode == "distinct":\n                        assert center[2:] == ("diamond", "yellow")\n                        assert {left[2:], right[2:]} == {("square", "green"), ("triangle", "purple")}\n                        assert len({row[2] for row in layout}) == len({row[3] for row in layout}) == 3\n                    else:\n                        assert all(row[2:] == ("square", "green") for row in layout)\n                seen = alcanzables(raw)\n                assert corners | button_access <= seen\n                assert raw.grid.get(8, 3).__class__.__name__ == "Button"\n                assert all(raw.grid.get(8, y) is None for y in range(4, 9))\n\n                for action in (0, 1, 0, 1):\n                    env.step(action)\n                    assert figuras(raw) == layout\n                # Every approach to the button remains usable.\n                for pos, direction in (((8, 4), 3), ((8, 2), 1), ((7, 3), 0), ((9, 3), 2)):\n                    raw.agent_pos, raw.agent_dir = pos, direction\n                    obs, _, _, _ = env.step(module.FORWARD)\n                    assert obs["log_presses"] == 1 and raw.cookie.cur_pos is not None\n                    cookies.append(tuple(raw.cookie.cur_pos))\n                    assert figuras(raw) == layout\n                spawner = raw.spawner\n                for corner in sorted(corners):\n                    raw.spawner = lambda pos=corner: pos\n                    raw.spawn_cookie()\n                    x, y = corner\n                    for direction, (dx, dy) in enumerate(((1, 0), (0, 1), (-1, 0), (0, -1))):\n                        pos = (x - dx, y - dy)\n                        if pos in seen and raw.grid.get(*pos) is None:\n                            raw.agent_pos, raw.agent_dir = pos, direction\n                            _, reward, _, _ = env.step(module.FORWARD)\n                            assert reward == 1, "La galleta no se puede recoger"\n                            break\n                    else:\n                        raise AssertionError("No hay acceso a una esquina")\n                    assert figuras(raw) == layout\n                raw.spawner = spawner\n                layouts.append(layout)\n        finally:\n            env._env.close()\n        return layouts, cookies\n\n    for view in (3, 5):\n        for seed in (0, 7):\n            _, no_cookies = secuencia("none", view, seed)\n            same, same_cookies = secuencia("same", view, seed)\n            distinct, distinct_cookies = secuencia("distinct", view, seed)\n            assert [tuple(row[:2] for row in layout) for layout in same] == [\n                tuple(row[:2] for row in layout) for layout in distinct\n            ], "same y distinct deben compartir geometría"\n            assert no_cookies == same_cookies == distinct_cookies, "Cambió el sorteo de galletas"\n            for room in range(3):\n                assert len({layout[room][:2] for layout in distinct}) > 1\n            for room in (0, 2):\n                assert {layout[room][2] for layout in distinct} == {"square", "triangle"}\n            assert secuencia("distinct", view, seed) == (distinct, distinct_cookies)\n            print(f"OK: vista {view}×{view}, semilla {seed}, 24 episodios por condición y repetición")\n\n    # Exhaust every possible placement, including combinations not drawn above.\n    env = module.Cookie("partial_hall5_view5", obstacles="distinct", seed=0)\n    try:\n        env.reset()\n        raw = env._env.unwrapped\n        original = figuras(raw)\n        objects = [raw.grid.get(*row[:2]) for row in original]\n        for row in original:\n            raw.grid.set(*row[:2], None)\n        count = 0\n        for positions in product(sorted(side_positions[0]), sorted(center_positions), sorted(side_positions[1])):\n            for pos, obj in zip(positions, objects):\n                raw.grid.set(*pos, obj)\n            assert corners | button_access <= alcanzables(raw)\n            for pos in positions:\n                raw.grid.set(*pos, None)\n            count += 1\n        assert count == 18 ** 3\n        print(f"OK: las {count} distribuciones posibles mantienen todas las casillas transitables conectadas")\n    finally:\n        env._env.close()\n    print("Verificado: tres figuras, colores distintos, posiciones variables y objetivos accesibles.")\n\n\nif __name__ == "__main__":\n    if len(sys.argv) > 1:\n        path = Path(sys.argv[1]).resolve()\n        spec = importlib.util.spec_from_file_location("cookie_semana25", path)\n        module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(module)\n    else:\n        from envs import cookie as module\n    verificar(module)\n'
subprocess.run([str(PYTHON), "-c", comprobacion], cwd=REPO, check=True)


In [ ]:
#@title Vista previa: tres episodios y las tres condiciones
from IPython.display import Image, display

vista_previa = 'from PIL import Image, ImageDraw, ImageFont\nfrom envs.cookie import Cookie\n\nmodes = ("none", "same", "distinct")\nnames = ("Sin figuras", "Tres cuadrados", "Tres formas y colores")\ncanvas = Image.new("RGB", (3 * 272, 3 * 220 + 24), "white")\ndraw = ImageDraw.Draw(canvas)\nfont = ImageFont.load_default(size=13)\nfor column, (mode, name) in enumerate(zip(modes, names)):\n    draw.text((column * 272 + 4, 4), name, font=font, fill="black")\n    env = Cookie("partial_hall5_view3", obstacles=mode, seed=0)\n    env._env.unwrapped.tile_size = 16\n    try:\n        for episode in range(3):\n            env.reset()\n            y = 24 + episode * 220\n            draw.text((column * 272 + 4, y), f"Episodio {episode + 1}", font=font, fill="black")\n            frame = env._env.unwrapped.get_frame(highlight=False, tile_size=16)\n            canvas.paste(Image.fromarray(frame), (column * 272, y + 20))\n    finally:\n        env._env.close()\ncanvas.save("/content/obstaculos_semana25_3x3.png")\n'
subprocess.run([str(PYTHON), "-c", vista_previa], cwd=REPO, check=True)
display(Image(filename="/content/obstaculos_semana25_3x3.png", width=816))


## 3. Elegir una condición

Usa `distinct` para los tres obstáculos de formas y colores diferentes. La semilla y los pasos se pueden ajustar aquí. Los resultados se guardan en una carpeta nueva de la semana del 25. El checkpoint anterior de dos obstáculos pertenece a otro entorno y no se carga en esta variante.


In [ ]:
MODO = "distinct" #@param ["none", "same", "distinct"]
SEMILLA = 0 #@param {type:"integer"}
PASOS = 500_000 #@param {type:"integer"}

TAREA = "cookie_partial_hall5_view3"
RAIZ = Path("/content/drive/MyDrive/galleta/semana25/tres_obstaculos_colores3x3")
MODOS = {"none": "Sin obstáculos", "same": "Tres cuadrados verdes", "distinct": "Tres formas y colores"}

def carpeta(modo, sampler=True):
    return RAIZ / f"{modo}{'_sampler1' if sampler else ''}_s{SEMILLA}"

assert MODO in MODOS
LOGDIR = carpeta(MODO)
print(MODOS[MODO], "· sampler 1 · semilla", SEMILLA)
print(LOGDIR)

def comando(pasos, logdir, *, prueba=False):
    return [
        str(PYTHON), "-u", "train.py",
        "env=cookie", f"env.task={TAREA}", f"env.obstacles={MODO}",
        f"env.steps={pasos}", "env.train_ratio=128", "env.size=[64,64]",
        f"env.eval_episode_num={1 if prueba else 5}", f"seed={SEMILLA}",
        "model.rep_loss=dreamer", "model.plan2explore=False", "trainer.random_policy=False", "buffer.reward_slices=1",
        "trainer.eval_every=25000", f"trainer.update_log_every={500 if prueba else 2500}",
        f"trainer.video_pred_log={not prueba}", f"trainer.save_buffer={not prueba}",
        "trainer.save_buffer_every=100000", "buffer.storage_device=cpu",
        f"buffer.max_size={pasos + 1 if prueba else 500_000}", f"logdir={logdir}",
    ]

## 4. Prueba corta opcional

Corre 1.500 pasos en disco local con la condición elegida. Sirve para comprobar el entrenamiento y que se registren las métricas, incluida la pérdida de imagen. La comparación de aprendizaje se hace en la sección siguiente.

In [ ]:
import json
import math
import tempfile

prueba = Path(tempfile.mkdtemp(prefix=f"galleta-semana25-3x3-{MODO}-sampler1-", dir="/content"))
subprocess.run(comando(1500, prueba, prueba=True), cwd=REPO, check=True)
filas = [json.loads(linea) for linea in (prueba / "metrics.jsonl").read_text().splitlines()]
assert any("episode/eval_cookie_in_front" in fila for fila in filas), "Faltan métricas de evaluación"
perdidas = [fila["train/loss/image"] for fila in filas if "train/loss/image" in fila]
assert perdidas and all(math.isfinite(p) for p in perdidas), "Revisa la pérdida de imagen y la GPU"
print("Prueba completada:", prueba)

## 5. Entrenar o retomar

Ejecuta esta celda para la condición seleccionada. La barra muestra los pasos y el tiempo estimado a medida que llegan los reportes de entrenamiento. Para detenerla, pulsa el botón de parar y espera a que termine de guardar en Drive.

In [ ]:
import re
from tqdm.auto import tqdm

paso_log = re.compile(r"^\[(\d+)\]")
paso_reinicio = re.compile(r"^Resuming from .* at step (\d+)")
paso_final = re.compile(r"^Training finished at step (\d+)\.")
args = comando(PASOS, LOGDIR)
proceso = subprocess.Popen(
    args, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

barra = tqdm(total=PASOS, desc=MODOS[MODO], unit="paso", smoothing=0)
barra.set_postfix_str("preparando")
detenido = False
try:
    while True:
        try:
            for linea in proceso.stdout:
                reinicio = paso_reinicio.match(linea)
                paso = paso_log.match(linea) or paso_final.match(linea)
                if reinicio:
                    barra.reset(total=PASOS)
                    barra.n = min(int(reinicio[1]), PASOS)
                    barra.initial = barra.last_print_n = barra.n
                    barra.refresh()
                if paso:
                    barra.update(max(0, min(int(paso[1]), PASOS) - barra.n))
                    if " progress " in linea:
                        continue
                if linea.startswith("Evaluating"):
                    barra.set_postfix_str("evaluando")
                elif "Saving the replay buffer" in linea:
                    barra.set_postfix_str("guardando el buffer")
                elif paso:
                    barra.set_postfix_str("")
                tqdm.write(linea.rstrip())
            estado = proceso.wait()
            break
        except KeyboardInterrupt:
            if not detenido and proceso.poll() is None:
                detenido = True
                proceso.terminate()
                barra.set_postfix_str("guardando")
                tqdm.write("Guardando el checkpoint y el buffer. Espera a que termine.")
finally:
    barra.close()

if estado:
    raise subprocess.CalledProcessError(estado, args)
print("Run guardado:", LOGDIR)

## 6. Comparar dos y tres obstáculos con sampler

Líneas continuas: esta variante de tres obstáculos. Líneas punteadas: el 3×3 con sampler de la semana del 24, si existe en Drive. Se compara la misma condición y semilla. La variante nueva cambia tanto el número de obstáculos como sus colores: una diferencia de rendimiento no permite atribuir el efecto a uno solo de esos cambios.


In [ ]:
import json
import matplotlib.pyplot as plt

metricas = {
    "episode/eval_score": "Galletas recogidas",
    "episode/eval_presses": "Pulsaciones del botón",
    "episode/eval_cookie_visible": "Decisiones con galleta visible",
    "episode/eval_cookie_in_front": "Decisiones con galleta justo delante",
}
raiz_anterior = Path("/content/drive/MyDrive/galleta/semana24/obstaculos_aleatorios3x3")
figura, ejes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, layout="constrained")
colores = dict(zip(MODOS, ("tab:gray", "tab:blue", "tab:orange")))
for modo, nombre in MODOS.items():
    for nuevo in (True, False):
        archivo = (carpeta(modo) if nuevo else raiz_anterior / f"{modo}_sampler1_s{SEMILLA}") / "metrics.jsonl"
        etiqueta = f"{modo} · {'3 obstáculos, colores nuevos' if nuevo else '2 obstáculos, verdes'}"
        if not archivo.exists():
            print(f"{etiqueta}: pendiente")
            continue
        filas = [json.loads(linea) for linea in archivo.read_text().splitlines() if linea.strip()]
        for eje, clave in zip(ejes.flat, metricas):
            puntos = sorted({fila["step"]: fila[clave] for fila in filas if clave in fila}.items())
            if puntos:
                pasos, valores = zip(*puntos)
                eje.plot(pasos, valores, marker="o" if nuevo else None,
                         linestyle="-" if nuevo else "--", color=colores[modo], label=etiqueta)
for eje, titulo in zip(ejes.flat, metricas.values()):
    eje.set_title(titulo)
    eje.set_xlabel("Pasos de entrenamiento")
    eje.set_ylabel("Promedio por partida")
    eje.grid(alpha=0.25)
    if eje.lines:
        eje.legend()
plt.show()


## 7. Ver jugar al actor

Graba una partida del checkpoint de esta condición y muestra el mapa junto a la imagen de 3×3 que recibe el agente. Cambiar la semilla del episodio cambia tanto el sorteo de las figuras como la secuencia de galletas.


In [ ]:
from IPython.display import Video, display

SEMILLA_EPISODIO = 0 #@param {type:"integer"}
if not (LOGDIR / "latest.pt").exists():
    print("Todavía no hay un checkpoint para esta condición y semilla.")
else:
    video = LOGDIR / f"agente_episodio{SEMILLA_EPISODIO}.mp4"
    subprocess.run([
        str(PYTHON), "watch.py", "--logdir", str(LOGDIR), "--out", str(video),
        "--seed", str(SEMILLA_EPISODIO),
    ], cwd=REPO, check=True)
    display(Video(str(video), embed=True, width=760))